<a href="https://colab.research.google.com/github/TheG0AT-0/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Part 0
import os

from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")


Client ready.


In [6]:
# Part 1.1
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response

result = ask_llm("Simple definition of microfinance")
print(result.choices[0].message.content)
print(result.usage)

Microfinance refers to the provision of small loans, savings, and other financial services to low-income individuals or groups, typically in developing countries, who lack access to traditional banking services.
CompletionUsage(completion_tokens=37, prompt_tokens=46, total_tokens=83, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.15208252, prompt_time=0.006713428, completion_time=0.115657987, total_time=0.122371415)


In [12]:
# Part 1.2
question = "Suggest a name for a savings product for market traders in Accra."

print(" For Temperature=0.0")
print("---------------------")
for i in range(5):
    r = ask_llm(question, temperature=0.0)
    print(i + 1, ".", r.choices[0].message.content, )
    print()

print()
print(" For Temperature=1.2")
print("---------------------")
for i in range(5):
    r = ask_llm(question, temperature=1.2)
    print(i + 1, ". ", r.choices[0].message.content, )
    print()

 For Temperature=0.0
---------------------
1 . Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to collect and save their earnings.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good fortune" or "prosperity" in the Akan language, which could be an attractive name for a savings product.
7. **Accra Trader's Fund**: This name is straightforward and emp

In [13]:
# Section 2
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [14]:
# Part 3.1
SUMMARY_PROMPT_V1 = "Summarize this: {letter}"
for lid in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V1.format(letter=LETTERS[lid])
    r = ask_llm(prompt)
    print(lid, ":", sep="")
    print(r.choices[0].message.content)
    print()

SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer. "
    "Summarize loan application letters factually and neutrally. "
    "Do not invent any detail. "
    "Keep the summary to 3-4 sentences."
)

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

for lid in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V2.format(letter=LETTERS[lid])
    r = ask_llm(prompt, system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
    print(lid, ":", sep="")
    print(r.choices[0].message.content)
    print()

L002:
Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but is optimistic it will improve after the festive season. He doesn't have collateral and is relying on good faith, promising to repay the loan when he can.

L006:
Kofi, a 22-year-old, is requesting a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on feedback from friends. He promises to repay the loan within a year, once the businesses are successful, and is offering his trustworthiness as assurance, as he has no collateral to offer.

L002:
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but h

In [15]:
# Part 3.2
EXTRACT_SYSTEM = (
    "You are a data extraction assistant for a microfinance loan officer. "
    "Extract information from loan application letters into strict JSON. "
    "Return ONLY a JSON object with EXACTLY these keys: "
    "applicant_name (string), amount_ghs (number), purpose (string), "
    "monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), "
    "repayment_months (number or null). "
    "If a field is not stated in the letter, use null. Do not guess."
)

FEWSHOT_LETTER = """Dear Sir,
My name is Ama Serwaa, a hairdresser in Tema. I am requesting GHS 6,000 to buy new
hairdryers and chairs. My shop currently earns about GHS 700 monthly. My brother will
guarantee the loan. I propose to repay over 10 months."""

FEWSHOT_JSON = """{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 6000,
  "purpose": "buy new hairdryers and chairs",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}"""

EXTRACT_PROMPT = """Here is an example.

Letter:
{fewshot_letter}

JSON:
{fewshot_json}

Now extract the same fields from this letter. Return ONLY the JSON object, nothing else.

Letter:
{letter}

JSON:"""

In [16]:
import json

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(
        fewshot_letter=FEWSHOT_LETTER,
        fewshot_json=FEWSHOT_JSON,
        letter=letter_text,
    )
    r = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=0, max_tokens=300)
    raw = r.choices[0].message.content.strip()

    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print("Warning: could not parse JSON for this letter.")
        print("Raw output was:", raw)
        return None

In [17]:
import pandas as pd

rows = []
for lid in LETTERS:
    fields = extract_fields(LETTERS[lid])
    if fields is not None:
        fields["letter_id"] = lid
        rows.append(fields)

df = pd.DataFrame(rows)
df = df[["letter_id", "applicant_name", "amount_ghs", "purpose",
         "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]]
df

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0
